In [1]:
import os
import getpass

if "GROQ_API_KEY" not in os.environ:
  os.environ["GROQ_API_KEY"]=getpass.getpass("Enter api key")

Enter api key··········


In [2]:
!pip install -q langgraph langchain-groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.0 MB/s eta 0:00:00


In [3]:
from langchain_groq import ChatGroq

In [4]:

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)


In [5]:
!pip install langchain_tavily langchain_community --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [6]:
if "TAVILY_API_KEY" not in os.environ:
  os.environ["TAVILY_API_KEY"]=getpass.getpass("Enter api key")


Enter api key··········


In [7]:
from langchain_tavily import TavilySearch

In [8]:
tavily_search=TavilySearch(max_results=2,
                           search_depth="advanced",
                           include_domains=["who.int"],
                           include_raw_content=True)

In [11]:
question="Whats is malaria"
search_results=tavily_search.invoke(question)

In [ ]:
search_results

In [14]:
from langchain_community.document_loaders import WebBaseLoader

In [15]:
scraped_docs=[]
urls = [
    result.get("url")
    for result in search_results["results"]
    if result.get("url")
]

print("Found URLs:", urls)


if not urls:
    print("No valid URLs in search results.")


for url in urls:

    try:

        print(f"Scraping {url}...")

        loader = WebBaseLoader(url)

        docs = loader.load()

        for doc in docs:
            doc.metadata["source"] = url

        scraped_docs.extend(docs)

    except Exception as e:

        print(f"Error scraping {url}: {e}")


print(f"Scraped {len(scraped_docs)} documents.")

Found URLs: ['https://www.who.int/news-room/fact-sheets/detail/malaria', 'https://www.who.int/health-topics/malaria']
Scraping https://www.who.int/news-room/fact-sheets/detail/malaria...
Scraping https://www.who.int/health-topics/malaria...
Scraped 2 documents.


In [18]:
scraped_docs[1]

Document(metadata={'source': 'https://www.who.int/health-topics/malaria', 'title': '\r\n\tMalaria\r\n', 'description': 'Malaria is a life-threatening disease caused by parasites that are transmitted to people through the bites of infected female Anopheles mosquitoes. It is preventable and curable. ', 'language': 'en'}, page_content="      \r\n\tMalaria\r\n                     \n   Skip to main content       \n\n\n \n\n\n\n\n\n\n\nGlobal\n\n\nRegions\n\n\n\n\n\n\n\nWHO Regional websites\n\n\n\n\n\n\n\nAfrica\n\n\n\n\n\nAmericas\n\n\n\n\n\nSouth-East Asia\n\n\n\n\n\nEurope\n\n\n\n\n\nEastern Mediterranean\n\n\n\n\n\nWestern Pacific\n\n\n\n\n\n\n\n\n\n   \n\n\n\n\n\n\n\n\n\n\n\n\n\nWhen autocomplete results are available use up and down arrows to review and enter to select.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\r\n        Select language\r\n    \n\nSelect language\nEnglish\nالعربية\n中文\nFrançais\nEspañol\n\n\n\n\n        \n            \n\n\n\n\n\n\n\n\n\n\n       \n\n\n\n\n\n\n\n\n\n\n\nHome\n\

In [22]:
from langchain_core.documents import Document
from typing import List
from typing_extensions import TypedDict

In [36]:
class GraphState(TypedDict):
  question: str
  documents:List[Document]
  sender:str


Your graph state contains:


question	User query

documents	Retrieved web/PDF/text documents

In [38]:
def web_search_node(state:GraphState)->GraphState:
    """Searches the web for information, then scrapes the content from the resulting URLs."""

    print("--- Calling Web Search Node ---")

    question = state["question"]

    tavily_search=TavilySearch(max_results=2,
                           search_depth="advanced",
                           include_domains=["who.int"],
                           include_raw_content=True)

    search_results=tavily_search.invoke(question)

    scraped_docs=[]
    urls = [
        result.get("url")
        for result in search_results["results"]
        if result.get("url")
    ]

    print("Found URLs:", urls)


    if not urls:
        print("No valid URLs in search results.")


    for url in urls:

        try:

            print(f"Scraping {url}...")

            loader = WebBaseLoader(url)

            docs = loader.load()

            for doc in docs:
                doc.metadata["source"] = url

            scraped_docs.extend(docs)

        except Exception as e:

            print(f"Error scraping {url}: {e}")


    print(f"Scraped {len(scraped_docs)} documents.")
    return {"documents":scraped_docs,"sender":"web_search_node"}



In [39]:
from langgraph.graph import START,END,StateGraph
workflow=StateGraph(GraphState)

In [40]:
workflow.add_node("web_search_node",web_search_node)

In [41]:
workflow.add_edge("web_search_node",END)

In [42]:
workflow.set_entry_point("web_search_node")#Same as an edge from start to this node

In [43]:
app=workflow.compile()

for output in app.stream(inputs):
This is same as app.invoke but invoke will give the output in the ui only when all the process are done but stream will display in the ui as when an single output is ready like continuous display will be done..it is faster



In [44]:
inputs = {
    "question": "What is the latest WHO report on malaria?"
}

for output in app.stream(inputs):

    for key, value in output.items():

        print(f"\n--- Output from Node: {key} ---")

        if 'documents' in value and value['documents']:

            print(f"Documents found: {len(value['documents'])}")
            print("-" * 30)

            for i, doc in enumerate(value['documents']):

                print(f"\n--- Document {i+1} ---")

                print(
                    f"Source: {doc.metadata.get('source', 'No source available')}"
                )

                # print("\nContent:")
                # print(doc.page_content)

        else:

            print("No documents found in this step.")

--- Calling Web Search Node ---
Found URLs: ['https://www.who.int/teams/global-malaria-programme/reports/world-malaria-report-2025', 'https://www.who.int/teams/global-malaria-programme/reports/world-malaria-report-2024']
Scraping https://www.who.int/teams/global-malaria-programme/reports/world-malaria-report-2025...
Scraping https://www.who.int/teams/global-malaria-programme/reports/world-malaria-report-2024...
Scraped 2 documents.

--- Output from Node: web_search_node ---
Documents found: 2
------------------------------

--- Document 1 ---
Source: https://www.who.int/teams/global-malaria-programme/reports/world-malaria-report-2025

--- Document 2 ---
Source: https://www.who.int/teams/global-malaria-programme/reports/world-malaria-report-2024
